# Create BARD Awards (United States - Israel Binational Agricultural Research and Development Fund)

Creates BARD awards from BARD's own publications (scraped by
`scripts/local/bard_to_s3.py`; robots.txt on bard-isus.com / bard-isus.org
allows all agents):

1. **Research projects 1979-2026 (1,598)** from the "Research by State" map
   (https://www.bard-isus.com/static/bardmap.html -> `ResearchforMaps/rbystate?id=<state>`):
   project number, award year, panel, duration, approved budget (USD), title,
   all investigators (initials, surname, institution). Deduped on project
   number across states. Includes the joint programmes by panel name
   (NIFA-BARD, Texas-BARD, Maryland-BARD, Cornell-BARD, MARD, CPS).
2. **Full project numbers** (`IS-5125-18R` style) from the annual "BARD
   Approved Projects" PDFs (2009-2024, current lists, NIFA-BARD, MARD) and the
   per-project final-report abstract pages (`FRAbst/{n}.htm`, which also give
   the description for 733 projects). 1,145 of 1,598 projects have a
   BARD-published number; 2 more resolve from an unambiguous citation form
   (OpenAlex works API); the other 451 (mostly 1979-1989 and the 2025-2026
   awards not yet in any list) ship a synthetic `BARD-{number}` key.
3. **Fellowships (training pathway, in scope):** Senior Research Fellowships
   1990-2024 (34, `FR-n-yy`), Graduate Student Fellowships 2007-2024 (35,
   `GS-n-yy`), Vaadia-BARD Postdoctoral Fellowship alumni 1986-2023 (275,
   no numbers published, synthetic `BARD-PD-{year}-{surname}-{given}`;
   display_name composed from fellow + host). No amounts for fellowships.

Not ingested: BARD workshops (no grant-level list) and the Food Security
Technology Accelerator. No filter applied: every published project/fellow is kept.

**`funder_award_id` (§2.1.1):** citing works write BARD numbers as
`IS-4744-14`, `IS-4744-14R`, `IS-4744-14 R`, `IS‐4744‐14` (unicode hyphens),
`BARD IS-...`. The bare `{PP}-{number}-{yy}` form (no R/C/CR/F/P suffix) is
the most frequent exact spelling in the citation stubs (444 rows vs 236 with
an attached suffix, 85 spaced), so we ship that; the full number with suffix
is kept in `bard_number_full` (raw table). 375 existing stub rows (156
distinct ids) on F4320308883 collapse onto these awards.

**Dates:** BARD publishes only the award year and duration in years, so
`start_date`/`end_date` are NULL (BHF precedent: no fabricated day/month);
`start_year` = award year, `end_year` = award year + duration.

**Twin:** OpenAlex also has F4320320627 "BARD" (GB, ROR 031pjfg83 = Bard
(United Kingdom), the medical-device company), whose stubs are almost all BARD
fund numbers (IS-/US-...; 39 stub rows match this ingest). We ship under the
fund's id F4320308883 only.

**Prerequisites:** run `scripts/local/bard_to_s3.py` first. It uploads
`s3://openalex-ingest/awards/bard/bard_projects.parquet` (§1.4 shrink guard).

**Funder details (F4320* path):**
- funder_id: `4320308883`
- display_name: United States - Israel Binational Agricultural Research and Development Fund
- ror_id / doi: from `openalex.funders.funders` (ror 04e98j339, doi 10.13039/100006031)

**Priority:** `592` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.bard_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/bard/bard_projects.parquet`;

In [ ]:
%sql
SELECT record_type, award_id_source, COUNT(*) as n
FROM openalex.awards.bard_raw
GROUP BY 1, 2 ORDER BY 1, 2;

In [ ]:
%sql
DESCRIBE openalex.awards.bard_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.bard_raw LIMIT 5;

## Step 1.6: Funder existence check (F4320* path)
F4320308883 must resolve to exactly 1 row in `openalex.funders.funders`.
If 0 rows, STOP and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320308883;

## Step 2: Create BARD Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.bard_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320308883  -- US-Israel BARD Fund
),

people AS (
    -- investigators in the order BARD lists them (lead first).
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    SELECT
        funder_award_id,
        TRANSFORM(
            from_json(investigators_json,
                      'ARRAY<STRUCT<given_name:STRING, family_name:STRING, institution:STRING, role:STRING>>'),
            p -> named_struct(
                'given_name', NULLIF(TRIM(p.given_name), ''),
                'family_name', NULLIF(TRIM(p.family_name), ''),
                'orcid', CAST(NULL AS STRING),
                'role_start', CAST(NULL AS DATE),
                'affiliation', CASE WHEN NULLIF(TRIM(p.institution), '') IS NOT NULL THEN named_struct(
                    'name', TRIM(p.institution),
                    'country', CAST(NULL AS STRING),
                    'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
                ) END
            )
        ) AS investigators
    FROM openalex.awards.bard_raw
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    NULLIF(TRY_CAST(g.amount AS DOUBLE), 0) as amount,
    CASE WHEN NULLIF(TRY_CAST(g.amount AS DOUBLE), 0) IS NOT NULL THEN 'USD' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    g.funding_type as funding_type,
    NULLIF(TRIM(g.funder_scheme), '') as funder_scheme,
    'bard_funded_research' as provenance,
    CAST(NULL AS DATE) as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.award_year AS INT) as start_year,
    TRY_CAST(g.award_year AS INT) + TRY_CAST(g.duration_years AS INT) as end_year,
    try_element_at(p.investigators, 1) as lead_investigator,
    try_element_at(p.investigators, 2) as co_lead_investigator,
    CASE WHEN size(p.investigators) > 0 THEN p.investigators END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.bard_raw g
JOIN people p ON p.funder_award_id = g.funder_award_id
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

In [ ]:
%sql
-- Shape check (§2.1.1): research projects ship "{PP}-{number}-{yy}" without the
-- R/C/CR/F/P suffix, or the synthetic BARD-{number}; fellowships FR-/GS-n-yy or BARD-PD-.
-- Expect 0 rows.
SELECT funder_award_id, funder_scheme
FROM openalex.awards.bard_awards
WHERE NOT (funder_award_id RLIKE '^[A-Z]{2}-[0-9]{1,4}-[0-9]{2}$'
           OR funder_award_id RLIKE '^BARD-[0-9]+$'
           OR funder_award_id RLIKE '^BARD-PD-[0-9]{4}-[a-z0-9]+-[a-z0-9]*$');

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'bard_funded_research' AND priority = 592;

-- Insert into openalex_awards_raw with priority.
-- Priority 592: direct-from-funder ingest of BARD's own project lists.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500), so 592 outranks
-- the acknowledgement shells (priority 0) that share these award ids.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    592 as priority
FROM openalex.awards.bard_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_bard_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.bard_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_year, end_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name, landing_page_url
FROM openalex.awards.bard_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, funding_type, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_usd
FROM openalex.awards.bard_awards
GROUP BY 1, 2 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_usd
FROM openalex.awards.bard_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.bard_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.bard_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 / 6.7 coverage (expect 100% title/PI, ~82% amount (fellowships have none), ~38% description).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_year) as has_start_year,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.bard_awards;

In [ ]:
%sql
-- Overlap with existing acknowledgement stubs (priority 0) that these awards upgrade.
SELECT c.provenance, COUNT(*) as stub_rows, COUNT(b.id) as upgraded_by_bard_ingest
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.bard_awards b ON b.id = c.id
WHERE c.funder_id = 4320308883 AND c.priority = 0
GROUP BY c.provenance;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'bard_funded_research'
GROUP BY provenance, priority;